# FER2013 ResNeXt50 Separated Experiment Notebook

This version is designed for screenshots and reporting.

Run flow:
1. Run setup cells once.
2. Select one experiment ID.
3. Run **training cell only**.
4. Screenshot training output/curves.
5. Run train/validation/test evaluation cells separately.
6. Screenshot each result and confusion matrix.
7. Append result to comparison table.
8. Change experiment ID and repeat.


In [ ]:
# --- 1. Install / Import libraries ---
!pip -q install kagglehub

import os
import copy
import time
import gc
from collections import Counter

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import datasets, models, transforms
from torchvision.models import ResNeXt50_32X4D_Weights

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print("PyTorch version:", torch.__version__)
print("TorchVision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cpu":
    print("WARNING: You are using CPU. Go to Runtime > Change runtime type > T4 GPU.")


In [ ]:
# --- 2. Download FER2013 directly inside Google Colab using KaggleHub ---
# This downloads into Colab temporary cloud storage, NOT your laptop storage.

path = kagglehub.dataset_download("astraszab/facial-expression-dataset-image-folders-fer2013")
print("Downloaded dataset path:", path)

# Search automatically for folder that contains train + test/validation folders.
def find_fer_base(root):
    for current, dirs, files in os.walk(root):
        lower_dirs = [d.lower() for d in dirs]
        has_train = "train" in lower_dirs
        has_test_or_val = any(x in lower_dirs for x in ["test", "validation", "valid"])
        if has_train and has_test_or_val:
            return current
    return None

base_dir = find_fer_base(path)

if base_dir is None:
    raise FileNotFoundError("FER2013 dataset folder not found. Check the downloaded KaggleHub path.")

print("FER2013 base_dir:", base_dir)
print("Folders:", os.listdir(base_dir))


In [ ]:
# --- 3. Dataset paths and class names ---
train_dir = os.path.join(base_dir, "train")

test_dir = None
for possible_name in ["test", "validation", "valid"]:
    possible_path = os.path.join(base_dir, possible_name)
    if os.path.exists(possible_path):
        test_dir = possible_path
        break

if test_dir is None:
    raise FileNotFoundError("No test/validation folder found.")

print("train_dir:", train_dir)
print("test_dir:", test_dir)
print("Train classes:", os.listdir(train_dir))
print("Test classes:", os.listdir(test_dir))


In [ ]:
# --- 4. ResNeXt50 input settings + transforms ---
# FER2013 is grayscale 48x48.
# ResNeXt50 pretrained on ImageNet expects 3-channel 224x224 input.
# Grayscale(num_output_channels=3) converts each FER image into 3 channels.

weights = ResNeXt50_32X4D_Weights.DEFAULT
imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std

basic_train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((232, 232), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

no_aug_train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

strong_train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((232, 232), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    transforms.RandomErasing(p=0.25),
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class_check_dataset = datasets.ImageFolder(train_dir, transform=val_test_transform)
class_names = class_check_dataset.classes
num_classes = len(class_names)

print("Class names:", class_names)
print("Number of classes:", num_classes)


In [ ]:
# --- 5. Check class imbalance ---
targets = [label for _, label in class_check_dataset.samples]
class_counts = Counter(targets)

print("Class distribution:")
for i, class_name in enumerate(class_names):
    print(f"{i} = {class_name}: {class_counts[i]}")

plt.figure(figsize=(8, 4))
plt.bar([class_names[i] for i in range(num_classes)], [class_counts[i] for i in range(num_classes)])
plt.xticks(rotation=45)
plt.title("FER2013 Training Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.show()


In [ ]:
# --- 6. Experiment list ---
# For reporting, run ONE experiment at a time by changing CURRENT_EXP_ID later.

experiments = [
    {"id": 0, "name": "Baseline", "epochs": 10, "lr": 0.0001, "batch_size": 32, "class_weights": True,  "augmentation": "basic",  "freeze_mode": "frozen"},
    {"id": 1, "name": "No Augmentation", "epochs": 10, "lr": 0.0001, "batch_size": 32, "class_weights": True,  "augmentation": "none",   "freeze_mode": "frozen"},
    {"id": 2, "name": "3 Epochs", "epochs": 3,  "lr": 0.0001, "batch_size": 32, "class_weights": True,  "augmentation": "basic",  "freeze_mode": "frozen"},
    {"id": 3, "name": "20 Epochs", "epochs": 20, "lr": 0.0001, "batch_size": 32, "class_weights": True,  "augmentation": "basic",  "freeze_mode": "frozen"},
    {"id": 4, "name": "Higher LR 0.001", "epochs": 10, "lr": 0.001,   "batch_size": 32, "class_weights": True,  "augmentation": "basic",  "freeze_mode": "frozen"},
    {"id": 5, "name": "Lower LR 0.00001", "epochs": 10, "lr": 0.00001, "batch_size": 32, "class_weights": True,  "augmentation": "basic",  "freeze_mode": "frozen"},
    {"id": 6, "name": "No Class Weights", "epochs": 10, "lr": 0.0001, "batch_size": 32, "class_weights": False, "augmentation": "basic",  "freeze_mode": "frozen"},
    {"id": 7, "name": "Partial Fine-Tuning", "epochs": 10, "lr": 0.0001, "batch_size": 32, "class_weights": True,  "augmentation": "basic",  "freeze_mode": "last_block"},
    {"id": 8, "name": "Batch Size 64", "epochs": 10, "lr": 0.0001, "batch_size": 64, "class_weights": True,  "augmentation": "basic",  "freeze_mode": "frozen"},
]

pd.DataFrame(experiments)


In [ ]:
# --- 7. Helper functions: dataloaders, model, training, evaluation ---

def get_train_transform(augmentation):
    if augmentation == "basic":
        return basic_train_transform
    if augmentation == "none":
        return no_aug_train_transform
    if augmentation == "strong":
        return strong_train_transform
    raise ValueError(f"Unknown augmentation mode: {augmentation}")


def create_dataloaders(batch_size, augmentation, val_ratio=0.2):
    train_transform = get_train_transform(augmentation)

    train_full_aug = datasets.ImageFolder(train_dir, transform=train_transform)
    train_full_clean = datasets.ImageFolder(train_dir, transform=val_test_transform)
    test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transform)

    val_size = int(len(train_full_aug) * val_ratio)
    train_size = len(train_full_aug) - val_size

    generator = torch.Generator().manual_seed(42)
    train_indices, valid_indices = random_split(range(len(train_full_aug)), [train_size, val_size], generator=generator)

    train_dataset = torch.utils.data.Subset(train_full_aug, train_indices.indices)
    valid_dataset = torch.utils.data.Subset(train_full_clean, valid_indices.indices)

    dataloaders = {
        "train": DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True),
        "valid": DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
        "test": DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
    }

    dataset_sizes = {
        "train": len(train_dataset),
        "valid": len(valid_dataset),
        "test": len(test_dataset),
    }

    train_targets = [train_full_aug.samples[i][1] for i in train_indices.indices]
    return dataloaders, dataset_sizes, train_targets


def build_resnext50(freeze_mode="frozen"):
    # This line proves it is ResNeXt50 32x4d, not a basic CNN.
    model = models.resnext50_32x4d(weights=weights)

    for param in model.parameters():
        param.requires_grad = False

    if freeze_mode == "frozen":
        pass
    elif freeze_mode == "last_block":
        for param in model.layer4.parameters():
            param.requires_grad = True
    elif freeze_mode == "unfrozen":
        for param in model.parameters():
            param.requires_grad = True
    else:
        raise ValueError(f"Unknown freeze_mode: {freeze_mode}")

    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model.to(device)


def create_criterion(use_class_weights, train_targets):
    if not use_class_weights:
        print("Class weights: OFF")
        return nn.CrossEntropyLoss()

    counts = Counter(train_targets)
    total = sum(counts.values())
    weights_list = []
    for i in range(num_classes):
        weights_list.append(total / (num_classes * counts[i]))

    class_weights_tensor = torch.tensor(weights_list, dtype=torch.float).to(device)
    print("Class weights:", class_weights_tensor)
    return nn.CrossEntropyLoss(weight=class_weights_tensor)


def train_model(model, dataloaders, dataset_sizes, criterion, optimizer, scheduler, num_epochs, verbose=3):
    """
    verbose=1: only epoch summary
    verbose=2: phase summary
    verbose=3: batch progress + epoch summary
    """
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    history = {"train_loss": [], "valid_loss": [], "train_acc": [], "valid_acc": []}

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        for phase in ["train", "valid"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0
            num_batches = len(dataloaders[phase])
            print(f"{phase.upper()} phase started: {num_batches} batches")

            for batch_idx, (inputs, labels) in enumerate(dataloaders[phase], start=1):
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data).item()

                if verbose >= 3:
                    if batch_idx == 1 or batch_idx == num_batches or batch_idx % 100 == 0:
                        current_loss = running_loss / (batch_idx * inputs.size(0))
                        current_acc = running_corrects / min(batch_idx * inputs.size(0), dataset_sizes[phase])
                        print(f"  {phase} batch {batch_idx}/{num_batches} | running loss: {current_loss:.4f} | running acc: {current_acc:.4f}")

            if phase == "train":
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects / dataset_sizes[phase]

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            if verbose >= 1:
                print(f"{phase.upper()} Epoch Result -> Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

            if phase == "valid" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                print(f"New best validation accuracy: {best_acc:.4f}")

    time_elapsed = time.time() - since
    print("\nTraining complete")
    print(f"Time: {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best validation Acc: {best_acc:.4f}")

    model.load_state_dict(best_model_wts)
    return model, history, time_elapsed, best_acc


def plot_training_curves(history, title):
    plt.figure(figsize=(7, 4))
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["valid_acc"], label="Validation Accuracy")
    plt.title(f"{title} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["valid_loss"], label="Validation Loss")
    plt.title(f"{title} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


def plot_confusion_matrix(cm, class_names, title):
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    thresh = cm.max() / 2.0 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], "d"),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()


def evaluate_split(model, loader, criterion, split_name, exp_name, show_report=True, show_confusion=True):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            total_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    print("=" * 70)
    print(f"{exp_name} - {split_name} Results")
    print("=" * 70)
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Weighted F1 Score: {weighted_f1:.4f}")

    if show_report:
        print("\nClassification Report:")
        print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    if show_confusion:
        plot_confusion_matrix(cm, class_names, f"{exp_name} - {split_name} Confusion Matrix")

    return {
        f"{split_name.lower()}_loss": avg_loss,
        f"{split_name.lower()}_acc": acc,
        f"{split_name.lower()}_weighted_f1": weighted_f1,
    }


def get_experiment_by_id(exp_id):
    for exp in experiments:
        if exp["id"] == exp_id:
            return exp
    raise ValueError(f"Experiment ID {exp_id} not found.")


In [ ]:
# --- 8. Select ONE experiment ---
# Change this number only, then run the training/evaluation cells below.
# Example: 0 = baseline, 1 = no augmentation, 2 = 3 epochs, etc.

CURRENT_EXP_ID = 0
CURRENT_EXP = get_experiment_by_id(CURRENT_EXP_ID)
CURRENT_EXP_NAME = f"Exp {CURRENT_EXP['id']} - {CURRENT_EXP['name']}"

print("Selected experiment:")
print(CURRENT_EXP)

# Keep results across experiments in this Colab session.
if "all_results" not in globals():
    all_results = []

CURRENT_METRICS = {}


In [ ]:
# --- 9. TRAIN ONLY for selected experiment ---
# This cell trains the model and prints epoch/batch progress.
# Use verbose=3 to see detailed running output during each epoch.

print("#" * 90)
print("TRAINING", CURRENT_EXP_NAME)
print("#" * 90)

CURRENT_DATALOADERS, CURRENT_DATASET_SIZES, CURRENT_TRAIN_TARGETS = create_dataloaders(
    batch_size=CURRENT_EXP["batch_size"],
    augmentation=CURRENT_EXP["augmentation"]
)

print("Dataset sizes:", CURRENT_DATASET_SIZES)

CURRENT_MODEL = build_resnext50(CURRENT_EXP["freeze_mode"])
CURRENT_CRITERION = create_criterion(CURRENT_EXP["class_weights"], CURRENT_TRAIN_TARGETS)

trainable_params = [p for p in CURRENT_MODEL.parameters() if p.requires_grad]
CURRENT_OPTIMIZER = optim.Adam(trainable_params, lr=CURRENT_EXP["lr"])
CURRENT_SCHEDULER = lr_scheduler.StepLR(CURRENT_OPTIMIZER, step_size=7, gamma=0.1)

CURRENT_MODEL, CURRENT_HISTORY, CURRENT_TIME, CURRENT_BEST_VALID_ACC = train_model(
    CURRENT_MODEL,
    CURRENT_DATALOADERS,
    CURRENT_DATASET_SIZES,
    CURRENT_CRITERION,
    CURRENT_OPTIMIZER,
    CURRENT_SCHEDULER,
    num_epochs=CURRENT_EXP["epochs"],
    verbose=3
)

print("Training cell finished. Now run the curve/evaluation cells separately.")


In [ ]:
# --- 10. Training curves only ---
# Screenshot these graphs for the training section.

plot_training_curves(CURRENT_HISTORY, CURRENT_EXP_NAME)


In [ ]:
# --- 11. TRAIN SET evaluation only ---
# This shows training accuracy, F1, classification report, and confusion matrix.

train_metrics = evaluate_split(
    CURRENT_MODEL,
    CURRENT_DATALOADERS["train"],
    CURRENT_CRITERION,
    "Train",
    CURRENT_EXP_NAME,
    show_report=True,
    show_confusion=True
)

CURRENT_METRICS.update(train_metrics)
CURRENT_METRICS


In [ ]:
# --- 12. VALIDATION SET evaluation only ---
# This shows validation accuracy, F1, classification report, and confusion matrix.

valid_metrics = evaluate_split(
    CURRENT_MODEL,
    CURRENT_DATALOADERS["valid"],
    CURRENT_CRITERION,
    "Valid",
    CURRENT_EXP_NAME,
    show_report=True,
    show_confusion=True
)

CURRENT_METRICS.update(valid_metrics)
CURRENT_METRICS


In [ ]:
# --- 13. TEST SET evaluation only ---
# This shows test accuracy, F1, classification report, and confusion matrix.

# For final reporting, test accuracy is usually the most important number.
test_metrics = evaluate_split(
    CURRENT_MODEL,
    CURRENT_DATALOADERS["test"],
    CURRENT_CRITERION,
    "Test",
    CURRENT_EXP_NAME,
    show_report=True,
    show_confusion=True
)

CURRENT_METRICS.update(test_metrics)
CURRENT_METRICS


In [ ]:
# --- 14. Add current experiment result to comparison table ---
# Run this after train/valid/test evaluation cells are complete.

result_row = {
    "experiment": CURRENT_EXP["id"],
    "name": CURRENT_EXP["name"],
    "epochs": CURRENT_EXP["epochs"],
    "learning_rate": CURRENT_EXP["lr"],
    "batch_size": CURRENT_EXP["batch_size"],
    "augmentation": CURRENT_EXP["augmentation"],
    "class_weights": CURRENT_EXP["class_weights"],
    "freeze_mode": CURRENT_EXP["freeze_mode"],
    "best_valid_acc_during_training": CURRENT_BEST_VALID_ACC,
    "time_minutes": CURRENT_TIME / 60,
}
result_row.update(CURRENT_METRICS)

# Replace old row if you rerun the same experiment.
all_results = [r for r in all_results if r["experiment"] != CURRENT_EXP["id"]]
all_results.append(result_row)
all_results = sorted(all_results, key=lambda x: x["experiment"])

results_df = pd.DataFrame(all_results)
results_df


In [ ]:
# --- 15. Final comparison table for report ---
# Run this after you finish several experiments.

columns_to_show = [
    "experiment", "name", "epochs", "learning_rate", "batch_size",
    "augmentation", "class_weights", "freeze_mode",
    "train_acc", "valid_acc", "test_acc",
    "train_loss", "valid_loss", "test_loss",
    "train_weighted_f1", "valid_weighted_f1", "test_weighted_f1",
    "best_valid_acc_during_training", "time_minutes"
]

final_table = pd.DataFrame(all_results)
if len(final_table) == 0:
    print("No experiment results yet. Run training/evaluation and add result first.")
else:
    final_table = final_table[columns_to_show].copy()
    for col in final_table.columns:
        if final_table[col].dtype == "float64":
            final_table[col] = final_table[col].round(4)
    display(final_table)
    final_table.to_csv("resnext50_fer2013_separated_experiment_results.csv", index=False)
    print("Saved: resnext50_fer2013_separated_experiment_results.csv")


In [ ]:
# --- 16. Optional: save current model ---
# Run only if you want to save the current trained model.

model_filename = f"resnext50_fer2013_exp_{CURRENT_EXP['id']}.pth"
torch.save(CURRENT_MODEL.state_dict(), model_filename)
print("Saved model as:", model_filename)
